<a href="https://colab.research.google.com/github/mjcowley/Python_Intro/blob/main/06-Putting-It-Together.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://raw.githubusercontent.com/mjcowley/Python_Intro/main/Images/qarg_bw_dark.png" alt="QUT Astrophysics Research Group" width="400"/>

# Part 6: Putting It Together

**QUT Astrophysics Research Group (QARG) Python Training**

Group leads: Dr Michael Cowley and Dr Rossana Ruggeri
Author: Michael Cowley
Last updated: 18 Oct 2026

## Overview

You now have all the pieces: Python basics, NumPy and Astropy, tables and FITS files, plotting, and statistics. This final part puts them together in three short galaxy evolution projects, each built around a real scientific question.

The projects give you **less guidance** than the earlier parts. Each one sets out the question, the background you need and a list of steps, and leaves the code to you. This is much closer to what research is like. When you get stuck, look back at the earlier parts, check the documentation, or open the hints. Every project has a full solution, but try to get as far as you can without it.

The projects are independent, so you can do them in any order. Together with the final section on working in the group, this notebook takes about **2 hours**.

| Project | Question | Main skills |
|---------|----------|-------------|
| 1. The stellar mass function | How many galaxies of each mass are there, and how has that changed? | Volumes, histograms, error bars, log plots |
| 2. Large-scale structure | Are galaxies spread evenly through space? | Fine binning, peak finding, cosmological distances, sky maps |
| 3. The ages of galaxies | When did quiescent galaxies form their stars? | Cosmology, interpolation, model assumptions |

## Setup

Load the ZFOURGE catalogue as in Parts 4 and 5, and set up the cosmology we've used throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.table import Table
from astropy.utils.data import download_file

DATA_URL = "https://raw.githubusercontent.com/mjcowley/Python_Intro/main/data/"


def get_data(filename):
    """Download a course data file (only the first time) and return its local path."""
    return download_file(DATA_URL + filename, cache=True)


cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

df = pd.read_csv(get_data("zfourge_cdfs.csv"), na_values=[-99])
eazy = Table.read(get_data("zfourge_cdfs_eazy.fits"), hdu="EAZY").to_pandas().replace(-99, np.nan)
fast = pd.read_csv(get_data("zfourge_cdfs.fout"), sep=r"\s+", comment="#", na_values=[-99],
                   names=["id", "z", "ltau", "metal", "lage", "Av", "lmass", "lsfr", "lssfr", "la2t", "chi2"])

cat = pd.merge(df, eazy[["id", "z_peak", "f_U", "f_V", "f_J"]], on="id")
cat = pd.merge(cat, fast[["id", "z", "lage", "lmass", "lsfr"]].rename(columns={"z": "z_best"}), on="id")

gal = cat[cat["use"] == 1].copy()
gal["u_v"] = -2.5 * np.log10(gal["f_U"] / gal["f_V"])
gal["v_j"] = -2.5 * np.log10(gal["f_V"] / gal["f_J"])
gal["quiescent"] = (gal["u_v"] > 1.3) & (gal["v_j"] < 1.6) & (gal["u_v"] > 0.88 * gal["v_j"] + 0.59)

print(f"{len(cat)} objects in the catalogue, {len(gal)} with use = 1")

---
## Project 1: The Stellar Mass Function

### The question

How many galaxies are there of each stellar mass, and how has that changed over cosmic time?

### Background

The **stellar mass function** (SMF), $\Phi(M)$, is the number density of galaxies per unit volume per unit (log) mass:

$$\Phi = \frac{N}{V \, \Delta\log_{10} M}$$

where $N$ is the number of galaxies in a mass bin of width $\Delta\log_{10} M$ (in dex) and $V$ is the **comoving volume** of the survey in that redshift range. It's usually quoted in units of Mpc$^{-3}$ dex$^{-1}$ and plotted as $\log_{10}\Phi$. The SMF is one of the most fundamental measurements in galaxy evolution, and every model of galaxy formation has to reproduce it.

The comoving volume between two redshifts, for a survey covering a solid angle $\Omega$ of the sky, is a fraction $\Omega / 4\pi$ of the full-sky volume. Astropy gives you the full-sky volume out to redshift $z$ with `cosmo.comoving_volume(z)`.

**Survey area.** We need the area of sky the `use = 1` galaxies cover. One simple way is to split the sky into small square cells, count how many cells contain at least one galaxy, and multiply by the area of a cell. Here's that method:

In [ ]:
cell = 0.5 / 60                                   # cell size in degrees (0.5 arcmin)
cos_dec = np.cos(np.radians(gal["dec"].mean()))   # RA degrees are shorter away from the equator

ix = np.floor((gal["ra"] - gal["ra"].min()) * cos_dec / cell).astype(int)
iy = np.floor((gal["dec"] - gal["dec"].min()) / cell).astype(int)
n_cells = len(set(zip(ix, iy)))                   # a set keeps only the unique (x, y) pairs

area = n_cells * (cell * u.deg) ** 2
print(f"Survey area: {area.to(u.arcmin**2):.0f} = {area.to(u.deg**2):.4f}")

### Steps

1. For each of the redshift bins $0.5 < z < 1$, $1 < z < 1.5$, $1.5 < z < 2$ and $2 < z < 2.5$, calculate the comoving volume of the survey in Mpc$^3$.
2. In each redshift bin, count galaxies in stellar mass bins 0.25 dex wide from $10^{8.5}$ to $10^{12}\,M_\odot$, using `z_best` and `lmass`, and convert the counts to $\Phi$.
3. Estimate the uncertainty on each point from Poisson statistics: the uncertainty on a count $N$ is $\sqrt{N}$.
4. Plot $\log_{10}\Phi$ against $\log_{10}M$ for all four redshift bins on one set of axes, with error bars and a sequential colour map. Leave out empty bins.
5. Store $\log_{10}\Phi$ for the $10.5 < \log_{10}M < 10.75$ bin at $0.5 < z < 1$ in `logphi_check`.
6. Describe how the mass function changes with redshift.

<details>
<summary><b>Hints</b></summary>

- Solid angle: `area.to(u.sr).value` gives the area in steradians, so the fraction of the sky is that divided by $4\pi$.
- `np.histogram(masses, bins=edges)` returns the counts and the bin edges.
- Taking the log of an error bar: if $\Phi$ has uncertainty $\sigma_\Phi$, then $\log_{10}\Phi$ has uncertainty $\sigma_\Phi / (\Phi \ln 10)$, as in Part 5.
- Masking empty bins: `good = counts > 0`.

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: The Stellar Mass Function
  </h1>
</div>

Work through the steps above.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(logphi_check - (-2.70)) < 0.05, "logphi_check should be about -2.70. Check your volume and bin width."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
z_bins = [(0.5, 1.0), (1.0, 1.5), (1.5, 2.0), (2.0, 2.5)]
edges = np.arange(8.5, 12.01, 0.25)
centres = (edges[:-1] + edges[1:]) / 2
dlogm = 0.25
sky_fraction = area.to(u.sr).value / (4 * np.pi)
cmap = plt.get_cmap("viridis")

fig, ax = plt.subplots(figsize=(7, 5))
for i, (z_lo, z_hi) in enumerate(z_bins):
    volume = (cosmo.comoving_volume(z_hi) - cosmo.comoving_volume(z_lo)).to(u.Mpc**3).value * sky_fraction
    in_bin = gal[(gal["z_best"] > z_lo) & (gal["z_best"] < z_hi)]
    counts, _ = np.histogram(in_bin["lmass"], bins=edges)

    phi = counts / volume / dlogm
    phi_err = np.sqrt(counts) / volume / dlogm
    good = counts > 0

    log_phi = np.log10(phi[good])
    log_phi_err = phi_err[good] / (phi[good] * np.log(10))
    ax.errorbar(centres[good], log_phi, yerr=log_phi_err, fmt="o-", ms=4, capsize=2,
                color=cmap(i / len(z_bins)), label=f"${z_lo} < z < {z_hi}$  (V = {volume:.2e} Mpc$^3$)")

    if z_lo == 0.5:
        logphi_check = np.log10(phi[np.argmin(np.abs(centres - 10.625))])

ax.set_xlabel(r"$\log_{10}(M_*/M_\odot)$")
ax.set_ylabel(r"$\log_{10}\Phi$ (Mpc$^{-3}$ dex$^{-1}$)")
ax.set_title("ZFOURGE CDFS stellar mass function")
ax.legend(fontsize=9)
plt.show()

print(f"log Phi at log M = 10.625, 0.5 < z < 1: {logphi_check:.2f}")
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

### What it shows

At every redshift there are far more low-mass galaxies than high-mass ones, with a sharp drop above about $10^{11}\,M_\odot$. Going from $z \sim 2.25$ to $z \sim 0.75$, the number density of massive galaxies rises by a factor of a few, as galaxies keep growing by forming stars and merging.

Before trusting the details, think about what could go wrong. These are exactly the questions a referee would ask:

- **Mass completeness.** At high redshift, low-mass galaxies are too faint to detect (the Malmquist bias from Part 3), so the low-mass end is underestimated there. Real analyses calculate a **mass completeness limit** at each redshift and only plot above it. Look for where your high-redshift curves flatten or turn down.
- **Cosmic variance.** CDFS covers only about 120 square arcminutes, so a single overdense or underdense structure can shift the whole mass function. That's why ZFOURGE observed three fields, and why you'll see the structure in Project 2.
- **Uncertainties.** Poisson errors are only part of the story. Photometric redshift and stellar mass errors scatter galaxies between bins.

### Going further

- Compare your results with the ZFOURGE mass functions of [Tomczak et al. (2014)](https://ui.adsabs.harvard.edu/abs/2014ApJ...783...85T), which used all three fields.
- Split the sample into quiescent and star-forming galaxies, and plot their mass functions separately. Which type dominates at high mass?

---
## Project 2: Large-Scale Structure

### The question

Are galaxies spread evenly through space, or do they cluster? Can we see structure in a small deep field like CDFS?

### Background

On large scales, the Universe has a "cosmic web" of walls, filaments and clusters surrounding huge, nearly empty voids. Galaxies trace this structure, so if you measure their redshifts precisely, they bunch up at particular redshifts, called **redshift spikes**, wherever the line of sight passes through a wall or cluster.

Measuring this clustering statistically, for example with the **two-point correlation function**, is one of the main ways cosmologists test the cosmological model and measure dark energy. Here we'll do something simpler and look for individual structures.

A small difference in redshift corresponds to a velocity difference along the line of sight of

$$\Delta v = \frac{c\,\Delta z}{1 + z}$$

and, ignoring the galaxies' own motions, a comoving distance of $\Delta z \times c / H(z)$, where $H(z)$ is the Hubble parameter at that redshift (`cosmo.H(z)` in Astropy).

### Steps

1. Using **all** objects in `cat` with a spectroscopic redshift (structure doesn't care about the `use` flag), plot a histogram of `z_spec` between 0 and 1.5 with bins 0.005 wide. Can you see spikes?
2. Find the redshift of the most populated bin, and store the bin centre in `z_spike`.
3. Select the galaxies within $\pm 0.005$ of `z_spike`. What velocity range does $\Delta z = 0.005$ correspond to at this redshift? How deep is the slice in comoving Mpc?
4. Work out the comoving width of the field at `z_spike`. *Hint: the field is about 0.2° across, and `cosmo.comoving_transverse_distance(z)` times an angle in radians gives a comoving distance.* Compare it with the depth of your slice.
5. Plot the sky positions (RA and Dec) of all galaxies with spectroscopic redshifts in grey, and the spike members on top in colour. Remember that RA increases to the **left** on the sky. Are the members concentrated anywhere?
6. Now make the same redshift histogram using the **photometric** redshifts (`z_peak`) of `use = 1` galaxies. Can you still see the spikes? How do they compare with the spectroscopic ones? What does this tell you about the redshift precision you need to study large-scale structure?

<details>
<summary><b>Hints</b></summary>

- `counts, edges = np.histogram(...)` then `np.argmax(counts)` gives the index of the biggest bin.
- `ax.invert_xaxis()` makes RA increase to the left.
- Use `astropy.constants.c`, or just $c = 299{,}792$ km/s, and remember `cosmo.H(z)` has units of km/s/Mpc.
- For comparison, the photo-z scatter from Part 5 is about $0.01(1 + z)$.

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: Redshift Spikes in CDFS
  </h1>
</div>

Work through the steps above.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(z_spike - 0.735) < 0.006, "z_spike should be at the biggest peak, near z = 0.73."
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
from astropy import constants as const

spec_all = cat[cat["z_spec"].notna() & (cat["z_spec"] > 0)]

bins = np.arange(0, 1.5001, 0.005)
counts, edges = np.histogram(spec_all["z_spec"], bins=bins)
i_peak = np.argmax(counts)
z_spike = (edges[i_peak] + edges[i_peak + 1]) / 2

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(spec_all["z_spec"], bins=bins, color="0.4")
ax.axvline(z_spike, color="tab:red", ls="--", lw=1, label=f"Biggest spike, z = {z_spike:.4f}")
ax.set_xlabel("Spectroscopic redshift")
ax.set_ylabel("Number per 0.005")
ax.legend()
plt.show()

dz = 0.005
members = spec_all[np.abs(spec_all["z_spec"] - z_spike) < dz]
dv = (const.c * dz / (1 + z_spike)).to(u.km / u.s)
depth = (2 * dz * const.c / cosmo.H(z_spike)).to(u.Mpc)
width = (cosmo.comoving_transverse_distance(z_spike) * (0.2 * u.deg).to(u.rad).value).to(u.Mpc)
print(f"{len(members)} members within ±{dz} of z = {z_spike:.4f}")
print(f"±{dz} in redshift is ±{dv:.0f}, a slice {depth:.1f} deep (comoving)")
print(f"The field is only about {width:.1f} wide at this redshift (comoving)")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(spec_all["ra"], spec_all["dec"], s=4, color="0.75", label="All spec-z")
ax.scatter(members["ra"], members["dec"], s=18, color="tab:red", label=f"z = {z_spike:.3f} ± {dz}")
ax.invert_xaxis()
ax.set_aspect(1 / np.cos(np.radians(-27.8)))      # so a degree of RA and Dec look the same length
ax.set_xlabel("RA (deg)")
ax.set_ylabel("Dec (deg)")
ax.legend(loc="lower left")
plt.show()

photo = gal[(gal["z_peak"] > 0) & (gal["z_peak"] < 1.5)]
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(photo["z_peak"], bins=bins, color="tab:blue", alpha=0.7)
ax.axvline(z_spike, color="tab:red", ls="--", lw=1)
ax.set_xlabel("Photometric redshift")
ax.set_ylabel("Number per 0.005")
plt.show()

# The two biggest spikes are still visible, because ZFOURGE's medium-band filters give unusually
# precise photo-zs (about 0.01(1 + z) = 0.017 here), but they're several times broader than in the
# spectroscopic redshifts, and the smaller spikes have disappeared.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

### What it shows

The spectroscopic redshifts are strongly clustered. The biggest spike, at $z \approx 0.735$, and a second one near $z \approx 0.67$ are well-known structures in CDFS ([Gilli et al. 2003](https://ui.adsabs.harvard.edu/abs/2003ApJ...592..721G)). Without them, you'd expect only a handful of galaxies in each 0.005 bin at these redshifts.

The field is only about 9 comoving Mpc across, much smaller than the roughly 28 Mpc depth of our redshift slice, so we're seeing a small patch of structures that extend well beyond the field. Not all of that depth is real distance, though. Galaxies in groups and clusters move at several hundred km/s relative to each other, and those motions add to their redshifts, stretching structures along the line of sight. These **redshift-space distortions** are a nuisance for mapping, but measuring them statistically in large surveys is a powerful test of gravity and dark energy, and a key part of the group's cosmology research.

The biggest spikes even show up in the photometric redshifts, thanks to ZFOURGE's medium-band filters, but they're several times broader and the smaller ones vanish. Typical broad-band photometric redshifts, with about three times the scatter, would wash them out completely. To map structure along the line of sight, you need spectroscopic redshifts, which is why the big cosmology surveys are spectroscopic.

This is also why the mass function in Project 1 has to be treated with care. A field this small can easily be dominated by a few structures.

### Going further

- Find the next few biggest spikes. Do the quiescent fraction or masses of galaxies in the spikes differ from galaxies at similar redshifts outside them? This is a first step towards studying how **environment** affects galaxy evolution.
- Read about the two-point correlation function, the standard statistical measure of clustering. The [Corrfunc](https://corrfunc.readthedocs.io/) package calculates it efficiently for large surveys.

---
## Project 3: The Ages of Galaxies

### The question

Quiescent galaxies have stopped forming stars. How long ago did they form their stars, and how does that compare to the age of the Universe at the time we see them?

### Background

FAST fits each galaxy's SED with stellar population models, and one of the outputs is `lage`, the base-10 log of the **age** of the stellar population in years: roughly, how long ago the galaxy started forming stars. Ages from SED fitting are notoriously uncertain, but the difference between star-forming and quiescent galaxies is large enough to see.

Since we see a galaxy at redshift $z$, its stars must have started forming at a time $t_\mathrm{form} = t_\mathrm{Universe}(z) - \mathrm{age}$ after the Big Bang, and that corresponds to a **formation redshift** $z_\mathrm{form}$.

To turn a time into a redshift, we need to invert `cosmo.age(z)`. A fast way to do this for thousands of galaxies is **interpolation**: calculate the age of the Universe on a fine grid of redshifts, then use `np.interp()` to read off the redshift for any age.

In [ ]:
z_grid = np.linspace(0, 20, 2001)
age_grid = cosmo.age(z_grid).to(u.Gyr).value       # age of the Universe, decreasing with redshift

# np.interp needs the x values to increase, so we reverse both arrays
def redshift_at_age(t_gyr):
    """Redshift at which the Universe had age t_gyr (in Gyr)."""
    return np.interp(t_gyr, age_grid[::-1], z_grid[::-1])


print(f"The Universe was 1 Gyr old at z = {redshift_at_age(1.0):.2f}")
print(f"and 3 Gyr old at z = {redshift_at_age(3.0):.2f}")

### Steps

1. Select galaxies with $\log_{10}(M_*/M_\odot) > 10$ and $0.5 < z_\mathrm{best} < 2.5$. Add a column `age_gyr` with the age in Gyr, and a column `t_universe` with the age of the Universe at each galaxy's redshift.
2. Plot age against redshift, with star-forming and quiescent galaxies in different colours, and draw the age of the Universe as a line. Use a log y axis.
3. Calculate the median age of quiescent and star-forming galaxies, and store the quiescent median in `median_age_q`.
4. Calculate the formation redshift of each quiescent galaxy, and plot a histogram of them. When did most of today's quiescent galaxies start forming their stars?
5. Is any galaxy older than the Universe at its redshift? Why not? Think about what this means for how far you can trust these ages.

<details>
<summary><b>Hints</b></summary>

- `10 ** gal["lage"] / 1e9` converts `lage` to Gyr.
- `cosmo.age(z)` accepts a whole column of redshifts at once. Use `.to(u.Gyr).value` to get plain numbers.
- You'll notice the ages fall on horizontal lines. That's FAST's model grid again, as in Part 4.

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    Exercise: When Did Galaxies Form?
  </h1>
</div>

Work through the steps above.

In [ ]:
# your code here

In [ ]:
# Run this cell to check your answer
assert abs(median_age_q - 1.58) < 0.1, "median_age_q should be about 1.6 Gyr. Did you select log M > 10 and 0.5 < z < 2.5?"
print("Looks good!")

<details>
<summary><b>Show a solution</b> (try it yourself first!)</summary>

```python
massive = gal[(gal["lmass"] > 10) & (gal["z_best"] > 0.5) & (gal["z_best"] < 2.5)].copy()
massive["age_gyr"] = 10 ** massive["lage"] / 1e9
massive["t_universe"] = cosmo.age(massive["z_best"]).to(u.Gyr).value

q = massive[massive["quiescent"]]
sf = massive[~massive["quiescent"]]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(sf["z_best"], sf["age_gyr"], s=6, alpha=0.4, color="tab:blue", label="Star-forming")
ax.scatter(q["z_best"], q["age_gyr"], s=10, alpha=0.7, color="tab:red", label="Quiescent")
z_line = np.linspace(0.5, 2.5, 100)
ax.plot(z_line, cosmo.age(z_line).to(u.Gyr).value, color="k", label="Age of the Universe")
ax.set_yscale("log")
ax.set_xlabel("Redshift")
ax.set_ylabel("Stellar population age (Gyr)")
ax.legend(loc="lower left")
plt.show()

median_age_q = q["age_gyr"].median()
print(f"Median age: quiescent {median_age_q:.2f} Gyr, star-forming {sf['age_gyr'].median():.2f} Gyr")

z_form = redshift_at_age(q["t_universe"] - q["age_gyr"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(z_form, bins=np.arange(0, 12.01, 0.5), color="tab:red", alpha=0.7)
ax.set_xlabel("Formation redshift")
ax.set_ylabel("Number of quiescent galaxies")
plt.show()
print(f"Median formation redshift of quiescent galaxies: {np.median(z_form):.1f}")

print("Older than the Universe:", (massive["age_gyr"] > massive["t_universe"] * 1.001).sum())
# None, because FAST only allows ages younger than the Universe at each galaxy's redshift. That's a
# sensible physical prior, but it means the age limit comes from the assumption, not the data. Galaxies
# piled up against the line are ones where the data would have preferred an even older age.
```

</details>

<div style="text-align: center;">
  <h1 style="background: #00407A; color: white; border: 0; padding: 20px 0; display: inline-block; width: 90%; box-sizing: border-box;">
    End of Exercise
  </h1>
</div>

### What it shows

Quiescent galaxies are typically about three times older than star-forming galaxies of similar mass. Most of them started forming their stars around $z \sim 1.5$ to 3, within the first few billion years of cosmic history, with a tail to much higher redshift. Massive galaxies formed early and fast, then stopped. This pattern is sometimes called **downsizing**.

No galaxy is older than the Universe, but not because the data say so: FAST doesn't allow it. Every model fit includes assumptions like this, sometimes called **priors**, and part of using any code's output responsibly is knowing what they are. Before using a catalogue in your project, read the paper that describes how it was made.

### Going further

- Plot age against stellar mass for quiescent galaxies. Are more massive galaxies older?
- Compare formation redshifts for quiescent galaxies observed at $z \sim 0.75$ and at $z \sim 2$.

---
## Working in the Group

Congratulations on getting this far! Before you start your project, here are some habits that will save you a lot of time and grief.

### Make your work reproducible

You, your supervisor, or a referee should be able to rerun your analysis and get the same answer.

- **Restart and run all** before you trust a notebook's results. Hidden state from cells you ran out of order, or deleted, is the most common source of notebook bugs.
- **Never edit raw data.** Keep downloaded data as it came, and do all cleaning in code.
- **Set random seeds**, record which version of each catalogue you used, and note when you downloaded it.
- **Move reusable code into functions.** When a notebook gets long, put functions you use again into a `.py` file and `import` them.
- **Comment the *why*, not the *what*.** `# exclude stars near the field edge, see Straatman+16 Sec 3` helps far more than `# filter rows`.

### Use version control

**Git** keeps a history of every change to your code, so you can always go back, and **GitHub** lets you share it and work with others. The basic cycle is:

```bash
git clone https://github.com/<user>/<repo>.git   # get a copy of a repository
git status                                       # see what's changed
git add my_analysis.ipynb                        # stage a change
git commit -m "Add mass completeness limits"     # save a snapshot with a message
git push                                         # send your commits to GitHub
```

Commit little and often, with messages that say what changed and why. [GitHub's Git guide](https://docs.github.com/en/get-started/getting-started-with-git) and the [Software Carpentry Git lesson](https://swcarpentry.github.io/git-novice/) are good places to start. Ask your supervisor whether your project should have its own repository.

### Find and cite the literature

- **[NASA ADS](https://ui.adsabs.harvard.edu/)** is the search engine for astronomy papers. Use it to find papers, follow citations and export references for LaTeX.
- **[arXiv astro-ph](https://arxiv.org/list/astro-ph/new)** is where new papers appear first, every weekday.
- **Cite the software you use.** Astropy, NumPy, SciPy, Matplotlib and pandas all ask to be cited in papers. Run `import astropy; astropy.__citation__` to see how.

### Get help

- When you're stuck, first read the error message from the bottom up (Part 1), then search for it. If you're still stuck after about half an hour, **ask**. Everyone in the group has been stuck on the same things, and asking early is a skill, not a weakness.
- Your supervisor and the rest of the group are your best resource. Resources, computing access and group information are on the [QARG Canvas site](https://canvas.qut.edu.au/courses/13966).
- AI assistants can be very useful for explaining errors and suggesting code. Check everything they produce, make sure you understand it, and talk to your supervisor about appropriate use in your project.

### Computing

- Set up Python on your own machine using the instructions in the [README](https://github.com/mjcowley/Python_Intro#readme).
- For bigger jobs, the group has access to QUT's eResearch HPC and Swinburne's OzSTAR. Access details are on the [QARG Canvas site](https://canvas.qut.edu.au/courses/13966).

### Where next?

The group's research code lives in [Astro_Code](https://github.com/mjcowley/Astro_Code). Now that you've finished this training, you'll be able to follow it:

| Notebook | What it does |
|----------|--------------|
| `ZFOURGE-VizieR-Import` | Downloads the full ZFOURGE catalogues from VizieR |
| `ZFOURGE-EAZY` | Runs EAZY photometric redshifts on ZFOURGE |
| `ZFOURGE-RestFrame-UVJ` | Calculates rest-frame colours and UVJ diagrams |
| `SDSS-Luminosity-Function` | Measures the galaxy luminosity function from SDSS |
| `SED-Attenuation-and-Emission` | Adds dust attenuation and emission to galaxy SEDs |

For building physical intuition, try the interactive [Astro_Demos](https://mjcowley.github.io/astro_demos/).

<div class="practical-end" style="
  background: #f5f5f5;
  border-radius: 12px;
  padding: 20px 24px;
  text-align: center;
  border: 1px solid #ddd;
  margin: 2rem 0;
">
  <p style="font-size: 1.5rem; margin: 0 0 8px; text-align: center;">🎓</p>
  <h1 style="margin: 0 0 4px; color: #111; text-align: center;">You've Finished the Training</h1>
  <p style="margin: 0; color: #555; font-size: 0.9rem; text-align: center;">
    Well done! You're ready to start your project. Welcome to the group.
  </p>
</div>

<style>
@media (prefers-color-scheme: dark) {
  .practical-end { background: #1e1e1e !important; border-color: #333 !important; }
  .practical-end h1 { color: #f0f0f0 !important; }
  .practical-end p  { color: #999     !important; }
}
</style>